In [3]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

# ── 1. تعريف المتغيرات الأساسية ──────────────────────────
le = LabelEncoder()

features_final = [
    'rank_diff', 'home_form', 'away_form', 'form_diff',
    'h2h', 'tournament_weight',
    'home_avg_scored', 'home_avg_conceded',
    'away_avg_scored', 'away_avg_conceded',
    'avg_scored_diff', 'avg_conceded_diff',
    'neutral'
]

# ── 2. تحميل بيانات التقييمات ─────────────────────────────
team_ratings = pd.read_csv('data/clean/team_ratings.csv')

home_ratings = team_ratings.rename(columns={
    'team': 'home_team', 'avg_ovr': 'home_avg_ovr',
    'avg_sho': 'home_avg_sho', 'avg_def': 'home_avg_def',
    'avg_pac': 'home_avg_pac', 'avg_dri': 'home_avg_dri',
})

away_ratings = team_ratings.rename(columns={
    'team': 'away_team', 'avg_ovr': 'away_avg_ovr',
    'avg_sho': 'away_avg_sho', 'avg_def': 'away_avg_def',
    'avg_pac': 'away_avg_pac', 'avg_dri': 'away_avg_dri',
})

# ── 3. تحميل بيانات المباريات ودمجها ─────────────────────
df_ea = pd.read_csv('data/clean/results_features.csv')
df_ea = df_ea.drop(columns=['shootout_winner.1', 'shootout_winner.2'], errors='ignore')
df_ea['neutral'] = df_ea['neutral'].astype(int)

df_ea = df_ea.merge(home_ratings, on='home_team', how='left')
df_ea = df_ea.merge(away_ratings, on='away_team', how='left')

# ── 4. حساب الفروقات ──────────────────────────────────────
df_ea['form_diff']         = df_ea['home_form'] - df_ea['away_form']
df_ea['avg_scored_diff']   = df_ea['home_avg_scored'] - df_ea['away_avg_scored']
df_ea['avg_conceded_diff'] = df_ea['home_avg_conceded'] - df_ea['away_avg_conceded']
df_ea['ovr_diff']          = df_ea['home_avg_ovr'] - df_ea['away_avg_ovr']
df_ea['sho_diff']          = df_ea['home_avg_sho'] - df_ea['away_avg_sho']
df_ea['def_diff']          = df_ea['home_avg_def'] - df_ea['away_avg_def']
df_ea['pac_diff']          = df_ea['home_avg_pac'] - df_ea['away_avg_pac']
df_ea['dri_diff']          = df_ea['home_avg_dri'] - df_ea['away_avg_dri']

# ── 5. ملء القيم الناقصة ──────────────────────────────────
for col in ['ovr_diff', 'sho_diff', 'def_diff', 'pac_diff', 'dri_diff']:
    df_ea[col] = df_ea[col].fillna(0)

# ── 6. تصفية البيانات ─────────────────────────────────────
df_ea = df_ea[df_ea['tournament_weight'] >= 2].copy()

features_ea = features_final + [
    'ovr_diff', 'sho_diff', 'def_diff', 'pac_diff', 'dri_diff'
]

df_ea = df_ea.dropna(subset=features_ea)

# ── 7. تجهيز X و y ───────────────────────────────────────
X_ea = df_ea[features_ea]
y_ea = le.fit_transform(df_ea['result'])

print(f"إجمالي المباريات للتدريب: {len(df_ea)}")
print(f"عدد الـ features: {len(features_ea)}")
print(f"توزيع النتائج:\n{df_ea['result'].value_counts()}")

إجمالي المباريات للتدريب: 3751
عدد الـ features: 18
توزيع النتائج:
result
 1    1753
-1    1150
 0     848
Name: count, dtype: int64


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

voting_final = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(
            n_estimators=1000, max_depth=10, min_samples_leaf=5,
            class_weight='balanced', random_state=42, n_jobs=-1
        )),
        ('xgb', XGBClassifier(
            n_estimators=1000, max_depth=5, learning_rate=0.02,
            subsample=0.8, colsample_bytree=0.7, min_child_weight=5,
            eval_metric='mlogloss', random_state=42, n_jobs=-1
        )),
        ('lr', Pipeline([
            ('scaler', StandardScaler()),
            ('lr', LogisticRegression(
                max_iter=5000, class_weight='balanced', random_state=42
            )),
        ])),
    ],
    voting='soft'
)

scores_final = cross_val_score(
    voting_final, X_ea, y_ea,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy'
)

print(f"Voting RF + XGBoost + LR (scaled):")
print(f"  كل fold: {[round(s*100,1) for s in scores_final]}")
print(f"  المتوسط: {round(scores_final.mean()*100, 2)}%")

Voting RF + XGBoost + LR (scaled):
  كل fold: [59.0, 61.1, 60.5, 60.0, 60.0]
  المتوسط: 60.12%


In [5]:
# نحذف التعادل ونجرب binary classification
df_binary = df_ea[df_ea['result'] != 0].copy()

X_binary = df_binary[features_ea]
y_binary = (df_binary['result'] == 1).astype(int)  # 1=فوز المضيف, 0=فوز الضيف

print(f"عدد المباريات بدون تعادل: {len(df_binary)}")
print(f"توزيع النتائج:\n{y_binary.value_counts()}")

voting_binary = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(
            n_estimators=1000, max_depth=10, min_samples_leaf=5,
            class_weight='balanced', random_state=42, n_jobs=-1
        )),
        ('xgb', XGBClassifier(
            n_estimators=1000, max_depth=5, learning_rate=0.02,
            subsample=0.8, colsample_bytree=0.7, min_child_weight=5,
            eval_metric='logloss', random_state=42, n_jobs=-1
        )),
        ('lr', Pipeline([
            ('scaler', StandardScaler()),
            ('lr', LogisticRegression(
                max_iter=5000, class_weight='balanced', random_state=42
            )),
        ])),
    ],
    voting='soft'
)

scores_binary = cross_val_score(
    voting_binary, X_binary, y_binary,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy'
)

print(f"\nVoting Binary (فوز أو خسارة فقط):")
print(f"  كل fold: {[round(s*100,1) for s in scores_binary]}")
print(f"  المتوسط: {round(scores_binary.mean()*100, 2)}%")

عدد المباريات بدون تعادل: 2903
توزيع النتائج:
result
1    1753
0    1150
Name: count, dtype: int64

Voting Binary (فوز أو خسارة فقط):
  كل fold: [79.7, 77.1, 81.2, 82.6, 79.8]
  المتوسط: 80.09%


In [6]:
# ==============================
# الموديل 2 — هل المباراة تعادل؟
# ==============================

# نحول النتيجة: 1=تعادل, 0=مش تعادل
y_draw = (df_ea['result'] == 0).astype(int)

print(f"توزيع التعادل:")
print(f"  تعادل: {y_draw.sum()} ({round(y_draw.mean()*100,1)}%)")
print(f"  مش تعادل: {(y_draw==0).sum()} ({round((y_draw==0).mean()*100,1)}%)")

# Voting للتعادل
draw_model = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(
            n_estimators=1000, max_depth=10, min_samples_leaf=5,
            class_weight='balanced', random_state=42, n_jobs=-1
        )),
        ('xgb', XGBClassifier(
            n_estimators=1000, max_depth=5, learning_rate=0.02,
            subsample=0.8, colsample_bytree=0.7, min_child_weight=5,
            eval_metric='logloss', random_state=42, n_jobs=-1
        )),
        ('lr', Pipeline([
            ('scaler', StandardScaler()),
            ('lr', LogisticRegression(
                max_iter=5000, class_weight='balanced', random_state=42
            )),
        ])),
    ],
    voting='soft'
)

scores_draw = cross_val_score(
    draw_model, X_ea, y_draw,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy'
)

print(f"\nموديل التعادل:")
print(f"  كل fold: {[round(s*100,1) for s in scores_draw]}")
print(f"  المتوسط: {round(scores_draw.mean()*100, 2)}%")

توزيع التعادل:
  تعادل: 848 (22.6%)
  مش تعادل: 2903 (77.4%)

موديل التعادل:
  كل fold: [70.8, 70.8, 74.5, 75.7, 73.3]
  المتوسط: 73.05%


In [7]:
# ==============================
# تدريب وحفظ الموديلين
# ==============================

# الموديل 1 — Binary (فوز/خسارة)
binary_final = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(
            n_estimators=1000, max_depth=10, min_samples_leaf=5,
            class_weight='balanced', random_state=42, n_jobs=-1
        )),
        ('xgb', XGBClassifier(
            n_estimators=1000, max_depth=5, learning_rate=0.02,
            subsample=0.8, colsample_bytree=0.7, min_child_weight=5,
            eval_metric='logloss', random_state=42, n_jobs=-1
        )),
        ('lr', Pipeline([
            ('scaler', StandardScaler()),
            ('lr', LogisticRegression(
                max_iter=5000, class_weight='balanced', random_state=42
            )),
        ])),
    ],
    voting='soft'
)
binary_final.fit(X_binary, y_binary)
print("✅ تم تدريب الموديل 1 (Binary)")

# الموديل 2 — Draw
draw_final = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(
            n_estimators=1000, max_depth=10, min_samples_leaf=5,
            class_weight='balanced', random_state=42, n_jobs=-1
        )),
        ('xgb', XGBClassifier(
            n_estimators=1000, max_depth=5, learning_rate=0.02,
            subsample=0.8, colsample_bytree=0.7, min_child_weight=5,
            eval_metric='logloss', random_state=42, n_jobs=-1
        )),
        ('lr', Pipeline([
            ('scaler', StandardScaler()),
            ('lr', LogisticRegression(
                max_iter=5000, class_weight='balanced', random_state=42
            )),
        ])),
    ],
    voting='soft'
)
draw_final.fit(X_ea, y_draw)
print("✅ تم تدريب الموديل 2 (Draw)")

# حفظ الموديلين
with open('model_binary.pkl', 'wb') as f:
    pickle.dump(binary_final, f)

with open('model_draw.pkl', 'wb') as f:
    pickle.dump(draw_final, f)

with open('features.pkl', 'wb') as f:
    pickle.dump(features_ea, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("\n✅ تم حفظ كل الموديلات!")
print(f"  model_binary.pkl — دقة 79.8%")
print(f"  model_draw.pkl   — دقة 73.4%")
print(f"  features.pkl     — {len(features_ea)} feature")

✅ تم تدريب الموديل 1 (Binary)
✅ تم تدريب الموديل 2 (Draw)

✅ تم حفظ كل الموديلات!
  model_binary.pkl — دقة 79.8%
  model_draw.pkl   — دقة 73.4%
  features.pkl     — 18 feature
